In [ ]:
from pathlib import Path
import json, sys, time, zipfile
import numpy as np
import pandas as pd

ARTIFACTS = Path('/kaggle/input/datasets/nicksonmwamsojo/rogii-v1-artifacts')
if not ARTIFACTS.exists():
    ARTIFACTS = Path('/kaggle/input/rogii-v1-artifacts')

COMP = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
if not COMP.exists():
    COMP = Path('/kaggle/input/rogii-wellbore-geology-prediction')

print('Artifacts:', ARTIFACTS, '| exists:', ARTIFACTS.exists())
print('Comp:     ', COMP,      '| exists:', COMP.exists())

_models_zip = ARTIFACTS / 'models.zip'
if not (ARTIFACTS / 'models').exists() and _models_zip.exists():
    with zipfile.ZipFile(_models_zip, 'r') as _z:
        _z.extractall(ARTIFACTS)

print('Artifacts contents:', sorted(p.name for p in ARTIFACTS.iterdir()))


In [ ]:
sys.path.insert(0, str(ARTIFACTS / 'rogii_src'))

from rogii.features.spatial import FormationPlaneKNN, DenseANCCImputer
from rogii.physics.jit import warm_up_jit
from rogii.inference import predict_wells

print('rogii loaded')
warm_up_jit()
print('JIT ready')


In [ ]:
train_hw_paths = sorted((COMP / 'train').glob('*__horizontal_well.csv'))
train_wids = [p.stem.replace('__horizontal_well', '') for p in train_hw_paths]
print(f'Training wells: {len(train_wids)}')

t0 = time.time()
FI = FormationPlaneKNN(train_wids, COMP / 'train')
print(f'FormationPlaneKNN: {len(FI.df)} wells  ({time.time()-t0:.1f}s)')

t0 = time.time()
DI = DenseANCCImputer(train_wids, COMP / 'train')
print(f'DenseANCCImputer: {len(DI.ancc):,} points  ({time.time()-t0:.1f}s)')


In [ ]:
test_hw_paths = sorted((COMP / 'test').glob('*__horizontal_well.csv'))
print(f'Test wells: {len(test_hw_paths)}')

# Load target mode from train_meta
import joblib
_meta = joblib.load(ARTIFACTS / 'train_meta.pkl')
target_mode = _meta.get('target_mode', 'delta')
print(f'Target mode: {target_mode}')

t0 = time.time()
result_df = predict_wells(
    hw_paths      = test_hw_paths,
    artifacts_dir = ARTIFACTS,
    fi_imputer    = FI,
    di_imputer    = DI,
    is_train      = False,
    target_mode   = target_mode,
)
print(f'Inference: {time.time()-t0:.0f}s  rows={len(result_df)}  wells={result_df["well"].nunique()}')
print(f'Pred stats: mean={result_df["pred"].mean():.2f}  std={result_df["pred"].std():.2f}')


In [ ]:
sample_sub = pd.read_csv(COMP / 'sample_submission.csv')
sub = (
    sample_sub[['id']]
    .merge(result_df[['id', 'pred']].rename(columns={'pred': 'tvt'}), on='id', how='left')
)
nan_count = sub['tvt'].isna().sum()
if nan_count > 0:
    print(f'WARNING: {nan_count} IDs unmatched — check test well coverage!')
    sub['tvt'] = sub['tvt'].fillna(float(result_df['pred'].mean()))

out_path = Path('/kaggle/working/submission.csv')
sub.to_csv(out_path, index=False)
print(f'Rows: {len(sub):,}  NaN: {sub["tvt"].isna().sum()}')
print(f'Written to {out_path}')
sub.head()
